In [1]:
import warnings
warnings.filterwarnings('ignore')

import evaluate
import pandas as pd
import re
import json
import mlflow
from datasets import load_dataset, Dataset
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments, pipeline

## Data Preparation

In [2]:
# Loading the dataset
df = load_dataset("Tobi-Bueck/customer-support-tickets")['train'].to_pandas()
df

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51.0,Security,Outage,Disruption,Data Breach,None,None,None,None
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51.0,Account,Disruption,Outage,IT,Tech Support,None,None,None
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51.0,Product,Feature,Tech Support,None,None,None,None,None
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51.0,Billing,Payment,Account,Documentation,Feedback,None,None,None
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51.0,Product,Feature,Feedback,Tech Support,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61760,Assistance Needed for IFTTT Docker Integration,I am facing integration problems with IFTTT Do...,I would be happy to assist with the IFTTT Dock...,Problem,Technical Support,low,en,NaN,Integration,Disruption,Performance,IT,Tech Support,None,None,None
61761,Bitten um Unterstützung bei der Integration,"Sehr geehrte Kundenservice, ich möchte die Int...","Sehr geehrte [Name], vielen Dank für Ihren Kon...",Change,Technical Support,medium,de,NaN,Integration,Feature,Documentation,Tech Support,None,None,None,None
61762,None,"Hello Customer Support, I am inquiring about t...",We will send you detailed information on plans...,Request,Billing and Payments,low,en,NaN,Billing,Payment,Feature,Feedback,Sales,Lead,None,None
61763,Hilfe bei digitalen Strategie-Problemen,Die Qualität unserer digitalen Strategie-Bearb...,Um den digitalen Strategie-Impuls zu überprüfe...,Incident,Product Support,high,de,NaN,Feedback,Performance,IT,Tech Support,None,None,None,None


In [3]:
# Cleaning the text data

def clean_text(text):
    if pd.isna(text):
        return ''

    text = re.sub(r'\\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df['bert_body'] = df['body'].apply(clean_text)
df['bert_subject'] = df['subject'].apply(clean_text)

In [4]:
# Combining subject and body for BERT input

df = df[['bert_subject', 'bert_body', 'queue']]

def combine_subject_body(row):
    subject = row['bert_subject']
    body = row['bert_body']
    if subject:
        return subject + '. ' + body
    else:
        return body


df['subject_body'] = df.apply(combine_subject_body, axis=1)
df = df[['subject_body', 'queue']]
df

,subject_body,queue
0,Wesentlicher Sicherheitsvorfall. Sehr geehrtes...,Technical Support
1,Account Disruption. Dear Customer Support Team...,Technical Support
2,Query About Smart Home System Integration Feat...,Returns and Exchanges
3,Inquiry Regarding Invoice Details. Dear Custom...,Billing and Payments
4,Question About Marketing Agency Software Compa...,Sales and Pre-Sales
...,...,...
61760,Assistance Needed for IFTTT Docker Integration...,Technical Support
61761,Bitten um Unterstützung bei der Integration. S...,Technical Support
61762,"Hello Customer Support, I am inquiring about t...",Billing and Payments
61763,Hilfe bei digitalen Strategie-Problemen. Die Q...,Product Support


In [5]:
"""# Encoding the target variable
label_dict_52 = defaultdict(int)

le = LabelEncoder()
df['queue_encoded'] = le.fit_transform(df['queue'])

for label in le.classes_:
    label_dict_52[label] = int(le.transform([label])[0])
"""

"# Encoding the target variable\nlabel_dict_52 = defaultdict(int)\n\nle = LabelEncoder()\ndf['queue_encoded'] = le.fit_transform(df['queue'])\n\nfor label in le.classes_:\n    label_dict_52[label] = int(le.transform([label])[0])\n"

In [6]:
"""# Saving the labels in JSON
with open('../data/class_labels/distilbert.json', 'w') as file:
    json.dump(label_dict_52, file, indent=4)"""

"# Saving the labels in JSON\nwith open('../data/class_labels/distilbert.json', 'w') as file:\n    json.dump(label_dict_52, file, indent=4)"

### Loading Label JSON file for mapping

In [7]:
with open('../data/class_labels/distilbert.json', 'r') as file:
    label2id = json.load(file)
    id2label = {value: key for key, value in label2id.items()}

df

,subject_body,queue
0,Wesentlicher Sicherheitsvorfall. Sehr geehrtes...,Technical Support
1,Account Disruption. Dear Customer Support Team...,Technical Support
2,Query About Smart Home System Integration Feat...,Returns and Exchanges
3,Inquiry Regarding Invoice Details. Dear Custom...,Billing and Payments
4,Question About Marketing Agency Software Compa...,Sales and Pre-Sales
...,...,...
61760,Assistance Needed for IFTTT Docker Integration...,Technical Support
61761,Bitten um Unterstützung bei der Integration. S...,Technical Support
61762,"Hello Customer Support, I am inquiring about t...",Billing and Payments
61763,Hilfe bei digitalen Strategie-Problemen. Die Q...,Product Support


### Undersampling dataset

In [8]:
# Undersampling the dataset
min_count = df['queue'].value_counts().min()

undersampled_df = (
    df.groupby('queue', group_keys=False)
      .sample(n=min_count, random_state=42)
      .reset_index(drop=True)
)

"""# Train test split
X_us = undersampled_df[['subject_body']]
y_us = undersampled_df['queue']
X_train_us, X_test_us, y_train_us, y_test_us = train_test_split(X_us, y_us, test_size=0.2, random_state=15, stratify=y_us)"""

us_data = Dataset.from_pandas(undersampled_df)


### Train test split 

In [9]:
us_train_test = us_data.train_test_split(test_size=0.2)
us_train = us_train_test['train']
us_test = us_train_test['test']

### Tokenization Process

In [10]:
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-multilingual-cased")

def tokenize_function(examples):
  # Pad/truncate each text to 512 tokens. Enforcing the same shape
  # could make the training faster.
  return tokenizer(
      examples["subject_body"],
      padding="max_length",
      truncation=True,
      max_length=128,
  )

seed = 22

# Tokenize the train and test datasets
train_tokenized = us_train.map(tokenize_function)
train_tokenized = train_tokenized.remove_columns(["subject_body"]).shuffle(seed=seed)

test_tokenized = us_test.map(tokenize_function)
test_tokenized = test_tokenized.remove_columns(["subject_body"]).shuffle(seed=seed)

Map: 100%|██████████| 2392/2392 [00:01<00:00, 1988.04 examples/s]


## Model Initialization

In [11]:
model = AutoModelForSequenceClassification.from_pretrained(
  "distilbert-base-uncased",
  num_labels=52,
  label2id=label2id,
  id2label=id2label,
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 7827.82it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Setting up Evaluation Metrics

In [12]:
# Define the target optimization metric
metric = evaluate.load("accuracy")


# Define a function for calculating our defined target optimization metric during training
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis=-1)
  return metric.compute(predictions=predictions, references=labels)

## Configuring the training environment

In [13]:
# Checkpoints will be output to this `training_output_dir`.
training_output_dir = "/tmp/sms_trainer"
training_args = TrainingArguments(
  output_dir=training_output_dir,
  eval_strategy='epoch',
  per_device_train_batch_size=8,
  per_device_eval_batch_size=8,
  logging_steps=8,
  num_train_epochs=3,
)

# Instantiate a `Trainer` instance that will be used to initiate a training run.
trainer = Trainer(
  model=model,
  args=training_args,
  train_dataset=train_tokenized,
  eval_dataset=test_tokenized,
  compute_metrics=compute_metrics,
)

## Creating MLFlow Experiment, Initiating MLFlow Run, and Monitoring the training progress

In [21]:
# Pick a name that you like and reflects the nature of the runs that you will be recording to the experiment.
mlflow.set_experiment("Support Queue Classification")
mlflow.set_tracking_uri("http://127.0.0.1:8080")
with mlflow.start_run() as run:
  trainer.train()

MlflowException: API request to http://127.0.0.1:8080/api/2.0/mlflow/experiments/get-by-name failed with exception HTTPConnectionPool(host='127.0.0.1', port=8080): Max retries exceeded with url: /api/2.0/mlflow/experiments/get-by-name?experiment_name=Support+Queue+Classification (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=8080): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))

## Creating a Pipeline with the Fine-Tuned Model

In [ ]:
tuned_pipeline = pipeline(
  task="text-classification",
  model=trainer.model,
  batch_size=8,
  tokenizer=tokenizer,
  device="cuda",
)